# Pipeline de datos

In [ ]:
!pip install pyspark pandas requests sqlalchemy pymysql matplotlib

^C


  Using cached boto3-1.43.12-py3-none-any.whl.metadata (6.6 kB)
  Using cached botocore-1.43.12-py3-none-any.whl.metadata (5.6 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.17.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached boto3-1.43.12-py3-none-any.whl (140 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
    --------------------------------------- 0.0/2.1 MB 660.6 kB/s eta 0:00:04
   ---- ----------------------------------- 0.2/2.1 MB 2.9 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 13.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/45.7 kB ? eta -:--:--
   ---------------------------------------- 45.7/45.7 kB 2.2 MB/s eta 0:00:00
Using cached botocore-1.43.12-py3-none-any.whl (15.0 MB)
   ----------------------------------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("VideoGames-MultiSource-ETL") \
    .getOrCreate()

In [18]:
import requests

API_KEY = "f32eaf20fa24527cd1d5471509391445"

city = "Medellin"

url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

response = requests.get(url)

data = response.json()

print(data)

{'cod': 401, 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}


In [8]:
with open("games_raw.json", "w") as file:
    json.dump(data, file)

In [9]:
games = data["results"]

df = pd.DataFrame(games)

KeyError: 'results'

In [ ]:
df_clean = df[
    [
        "name",
        "rating",
        "released",
        "metacritic"
    ]
]

In [ ]:
df_clean["rating"].hist()
plt.title("Distribución de ratings")
plt.show()

In [ ]:
df_clean.to_csv("games.csv", index=False)

In [ ]:
spark = SparkSession.builder \
    .appName("VideoGamesETL") \
    .getOrCreate()

In [ ]:
spark_df = spark.read.csv(
    "games.csv",
    header=True,
    inferSchema=True
)

In [ ]:
spark_df.groupBy("released").count().show()

In [ ]:
spark_df.write.mode("overwrite").parquet(
    "games_parquet"
)